# 🎬 Movie Review Sentiment Analysis
**Author:** Amaan Malik  
**Tools:** Python, Pandas, Scikit-learn, NLTK, Matplotlib, Seaborn  
**Dataset:** IMDB Movie Reviews Dataset (Kaggle)  

---

## Project Overview
This project builds an **NLP sentiment analysis model** that classifies movie reviews as **Positive** or **Negative** using text preprocessing and machine learning.

### Steps:
1. Load & Explore the Dataset (EDA)
2. Text Preprocessing (Cleaning, Lemmatization, Stopword Removal)
3. Feature Extraction (TF-IDF Vectorization)
4. Model Building (Naive Bayes, Logistic Regression, Linear SVM)
5. Model Evaluation (Accuracy, ROC-AUC, Confusion Matrix)
6. Predict Sentiment on Custom Reviews

## Step 1: Import Libraries

In [ ]:
# Core
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Text processing
import re
import string
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries imported successfully!')

## Step 2: Load the Dataset
> **Download:** https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews  
> Save as `IMDB Dataset.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')

print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head(5)

In [ ]:
# Check class balance
print('Sentiment Distribution:')
print(df['sentiment'].value_counts())
print(f'\nPositive: {(df["sentiment"]=="positive").sum()} | Negative: {(df["sentiment"]=="negative").sum()}')
print('Dataset is perfectly balanced — 50/50 split!')

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Sentiment distribution bar chart
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='sentiment', palette=['steelblue', 'coral'],
                   order=['positive', 'negative'])
plt.title('Sentiment Distribution', fontsize=14)
plt.xlabel('Sentiment')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Review length analysis
df['review_length'] = df['review'].apply(len)
df['word_count']    = df['review'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment, color in [('positive', 'steelblue'), ('negative', 'coral')]:
    subset = df[df['sentiment'] == sentiment]
    axes[0].hist(subset['review_length'], bins=60, alpha=0.6, color=color, label=sentiment)
    axes[1].hist(subset['word_count'],    bins=60, alpha=0.6, color=color, label=sentiment)

axes[0].set_title('Review Length (Characters)', fontsize=12)
axes[0].set_xlabel('Length'); axes[0].legend()
axes[1].set_title('Word Count per Review', fontsize=12)
axes[1].set_xlabel('Word Count'); axes[1].legend()

plt.tight_layout()
plt.show()

print('Average word count:')
print(df.groupby('sentiment')['word_count'].mean().round(1))

In [ ]:
# Sample reviews
print('POSITIVE REVIEW SAMPLE:')
print(df[df['sentiment']=='positive']['review'].iloc[0][:300], '...')
print('\nNEGATIVE REVIEW SAMPLE:')
print(df[df['sentiment']=='negative']['review'].iloc[0][:300], '...')

In [ ]:
# Word Cloud — Positive Reviews
pos_text = ' '.join(df[df['sentiment']=='positive']['review'])
wc_pos = WordCloud(width=900, height=400, background_color='white',
                   colormap='Blues', max_words=120).generate(pos_text)

plt.figure(figsize=(13, 5))
plt.imshow(wc_pos, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words — Positive Reviews', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Word Cloud — Negative Reviews
neg_text = ' '.join(df[df['sentiment']=='negative']['review'])
wc_neg = WordCloud(width=900, height=400, background_color='white',
                   colormap='Reds', max_words=120).generate(neg_text)

plt.figure(figsize=(13, 5))
plt.imshow(wc_neg, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words — Negative Reviews', fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Text Preprocessing

In [ ]:
stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def preprocess_text(text):
    """Full NLP preprocessing pipeline."""
    # Remove HTML tags (common in IMDB reviews)
    text = re.sub(r'<.*?>', '', text)
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove punctuation & numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords + lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

print('Preprocessing reviews... (this may take ~1 minute for 50k reviews)')
df['clean_review'] = df['review'].apply(preprocess_text)
print('Done!')

print('\nOriginal:')
print(df['review'].iloc[0][:200])
print('\nCleaned:')
print(df['clean_review'].iloc[0][:200])

In [ ]:
# Encode target labels
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print('Label encoding: positive=1, negative=0')

## Step 5: Train-Test Split & TF-IDF Vectorization

In [ ]:
X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train):,}')
print(f'Test samples:     {len(X_test):,}')

In [ ]:
# TF-IDF with unigrams + bigrams
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print('TF-IDF shape (train):', X_train_tfidf.shape)
print('TF-IDF shape (test): ', X_test_tfidf.shape)

## Step 6: Model Building
### Model 1 — Multinomial Naive Bayes

In [ ]:
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)
nb_preds = nb.predict(X_test_tfidf)
nb_proba = nb.predict_proba(X_test_tfidf)[:, 1]

print('=== Naive Bayes ===')
print(f'Accuracy: {accuracy_score(y_test, nb_preds):.4f}')
print(f'ROC-AUC:  {roc_auc_score(y_test, nb_proba):.4f}')

### Model 2 — Logistic Regression

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
lr_preds = lr.predict(X_test_tfidf)
lr_proba = lr.predict_proba(X_test_tfidf)[:, 1]

print('=== Logistic Regression ===')
print(f'Accuracy: {accuracy_score(y_test, lr_preds):.4f}')
print(f'ROC-AUC:  {roc_auc_score(y_test, lr_proba):.4f}')

### Model 3 — Linear SVM

In [ ]:
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_train_tfidf, y_train)
svm_preds = svm.predict(X_test_tfidf)

print('=== Linear SVM ===')
print(f'Accuracy: {accuracy_score(y_test, svm_preds):.4f}')

## Step 7: Evaluation & Visualization

In [ ]:
# Model comparison table
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    'Model':     ['Naive Bayes', 'Logistic Regression', 'Linear SVM'],
    'Accuracy':  [accuracy_score(y_test, nb_preds),
                  accuracy_score(y_test, lr_preds),
                  accuracy_score(y_test, svm_preds)],
    'Precision': [precision_score(y_test, nb_preds),
                  precision_score(y_test, lr_preds),
                  precision_score(y_test, svm_preds)],
    'Recall':    [recall_score(y_test, nb_preds),
                  recall_score(y_test, lr_preds),
                  recall_score(y_test, svm_preds)],
    'F1 Score':  [f1_score(y_test, nb_preds),
                  f1_score(y_test, lr_preds),
                  f1_score(y_test, svm_preds)]
}).round(4)

print(results.to_string(index=False))

In [ ]:
# Confusion matrices for all 3 models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, preds, title in zip(axes,
                             [nb_preds, lr_preds, svm_preds],
                             ['Naive Bayes', 'Logistic Regression', 'Linear SVM']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, preds):.3f}', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve — Naive Bayes vs Logistic Regression
plt.figure(figsize=(8, 6))

for proba, label, color in [
    (nb_proba, 'Naive Bayes',          'coral'),
    (lr_proba, 'Logistic Regression',  'steelblue')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.title('ROC Curve — Sentiment Analysis Models', fontsize=14)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Top positive & negative sentiment words (from Logistic Regression)
feature_names = tfidf.get_feature_names_out()
coefs = lr.coef_[0]

top_pos_idx = np.argsort(coefs)[-20:][::-1]
top_neg_idx = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(feature_names[top_pos_idx][::-1], coefs[top_pos_idx][::-1], color='steelblue')
axes[0].set_title('Top 20 Positive Sentiment Words', fontsize=12)
axes[0].set_xlabel('Coefficient')

axes[1].barh(feature_names[top_neg_idx], np.abs(coefs[top_neg_idx]), color='coral')
axes[1].set_title('Top 20 Negative Sentiment Words', fontsize=12)
axes[1].set_xlabel('|Coefficient|')

plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report — best model
print('=== Classification Report — Logistic Regression ===')
print(classification_report(y_test, lr_preds, target_names=['Negative', 'Positive']))

## Step 8: Predict Sentiment on Custom Reviews

In [ ]:
def predict_sentiment(review, model=lr, vectorizer=tfidf):
    """Predict sentiment of any movie review."""
    cleaned  = preprocess_text(review)
    vector   = vectorizer.transform([cleaned])
    pred     = model.predict(vector)[0]
    proba    = model.predict_proba(vector)[0] if hasattr(model, 'predict_proba') else None
    label    = '😊 POSITIVE' if pred == 1 else '😞 NEGATIVE'
    print(f'Review:     "{review[:80]}..."' if len(review) > 80 else f'Review: "{review}"')
    print(f'Sentiment:  {label}')
    if proba is not None:
        print(f'Confidence: Negative={proba[0]:.2%} | Positive={proba[1]:.2%}')
    print('-' * 65)

# Test on custom reviews
predict_sentiment("This movie was absolutely brilliant! The acting was superb and the story kept me on the edge of my seat throughout.")
predict_sentiment("Terrible film. Boring plot, bad acting, and a complete waste of two hours. I want my money back.")
predict_sentiment("It was okay. Some good scenes but the ending was disappointing and the pacing felt off.")
predict_sentiment("One of the best films I have ever seen. A masterpiece of storytelling and visual effects.")

## ✅ Conclusion

| Model               | Accuracy | ROC-AUC |
|---------------------|----------|---------|
| Naive Bayes         | ~85%     | ~92%    |
| Logistic Regression | ~89%     | ~96%    |
| Linear SVM          | ~89%     | —       |

**Key Findings:**
- **Logistic Regression and Linear SVM** achieve the best accuracy (~89%) on 50,000 reviews
- **TF-IDF with bigrams** captures phrase-level sentiment patterns like *not good*, *very bad*
- Removing **HTML tags** was a crucial preprocessing step specific to IMDB reviews
- Top positive words: *great, excellent, best, wonderful, brilliant*
- Top negative words: *worst, bad, terrible, boring, waste*

**Possible Improvements:**
- Use **VADER** (rule-based) for a fast baseline comparison
- Fine-tune a **BERT** model for state-of-the-art accuracy (~93%+)
- Handle negation more carefully (*not good* ≠ *good*)
- Deploy as a REST API using FastAPI or Flask